# 🍌 Swahili Fruit Classifier — Colab (fast, GPU, no full-dataset export)

This notebook fixes the original bug (model was accidentally trained to distinguish
"Training" vs "Test" folders instead of fruit types) and is built to run **fast**:

- Runs on Colab's free GPU (no phone-verification wall like Kaggle's accelerator toggle)
- Downloads the Fruits-360 zip **once**, then extracts **only** the ~16 fruit folders
  you actually need (never unpacks the full 7GB+ archive to disk)
- Uses MobileNetV2 transfer learning instead of training a CNN from scratch, so it
  converges in a handful of epochs instead of 60

**Before running:** Runtime → Change runtime type → Hardware accelerator → **GPU** (T4 is fine).


## Step 0 — Confirm GPU is actually on

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → GPU, then re-run this cell.")
else:
    print(f"✅ GPU detected: {gpus}")


## Step 1 — Kaggle auth

Kaggle's account page now shows **"Generate New Token"**, which gives you a single token
string to **copy** (not a `kaggle.json` file to download) — go to
Kaggle → your profile picture → **Settings** → **API** → **Generate New Token**, then
copy the token value.

Phone verification is only required for Kaggle's own GPU/TPU notebooks — it does **not**
block API/token access, so this works fine even without it.

Running the cell below will prompt you to paste the token (input is hidden, like a password).


In [ ]:
!pip install -q -U kaggle

import os
from getpass import getpass

token = getpass("Paste your Kaggle API token here (input hidden): ").strip()

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/access_token", "w") as f:
    f.write(token)
os.chmod("/root/.kaggle/access_token", 0o600)

os.environ["KAGGLE_API_TOKEN"] = token

print("✅ Kaggle credentials configured.")


## Step 2 — Download the dataset **zip only** (no auto-unzip)

This is one network transfer of the compressed archive. We do **not** call `--unzip`,
so nothing gets extracted yet — that's the part that was eating your disk/time before.


In [ ]:
!kaggle datasets download -d moltean/fruits -p /content --force


## Step 3 — Selectively extract ONLY the fruits we need

Instead of `extractall()` (which would unpack every size-variant and all 170+ classes —
the 7GB+ you saw before), we read the zip's file list in memory and pull out just the
members that belong to our 16 Swahili-mapped fruit folders, for the `Training` and `Test`
splits. Everything else in the archive is never written to disk.

`MAX_VARIETIES_PER_FRUIT` caps how many sub-varieties get merged into one Swahili label
(e.g. "Tomato" alone matches ~9 sub-folders in the raw dataset — Cherry Red, Heart, Maroon,
Yellow, not-ripened, etc.). Merging all of them makes some categories 5-10x bigger than
others and inflates total RAM/disk usage for no real classification benefit. Capping it
keeps the dataset (and your RAM) under control while still giving each fruit a healthy
amount of visual variety.


In [ ]:
import zipfile, os, shutil
from pathlib import Path

ZIP_PATH = "/content/fruits.zip"
OUTPUT_DIR = Path("/content/dataset_selected")
MAX_VARIETIES_PER_FRUIT = 3  # cap merged sub-folders per Swahili label

MATCH_RULES = {
    "Ndizi": "Banana",
    "Chungwa": "Orange",
    "Parachichi": "Avocado",
    "Kabichi": "Cabbage",
    "Tango": "Cucumber",
    "Biliganya": "Eggplant",
    "Zabibu": "Grape",
    "Embe": "Mango",
    "Nyanya": "Tomato",
    "Kiazi": "Potato",
    "Karoti": "Carrot",
    "Kitunguu": "Onion",
    "Tangawizi": "Ginger",
    "Peasi": "Pear",
    "Pilipili": "Pepper",
    "Nanasi": "Pineapple",
}

BASE_PREFIX = "fruits-360_100x100/fruits-360/"  # only the 100x100 variant — skip other sizes/meta

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

with zipfile.ZipFile(ZIP_PATH) as zf:
    all_names = zf.namelist()

    train_prefix = f"{BASE_PREFIX}Training/"
    all_folders = sorted({
        name[len(train_prefix):].split('/')[0]
        for name in all_names
        if name.startswith(train_prefix) and '/' in name[len(train_prefix):]
    })

    SEL_FRUITS = {
        category: sorted([f for f in all_folders if f.lower().startswith(keyword.lower())])[:MAX_VARIETIES_PER_FRUIT]
        for category, keyword in MATCH_RULES.items()
    }
    total_extracted = 0
    for split in ["Training", "Test"]:
        split_prefix = f"{BASE_PREFIX}{split}/"
        for swahili_label, matched_folders in SEL_FRUITS.items():
            dst_folder = OUTPUT_DIR / split / swahili_label
            dst_folder.mkdir(parents=True, exist_ok=True)

            copied = 0
            for folder_name in matched_folders:
                folder_prefix = f"{split_prefix}{folder_name}/"
                for name in all_names:
                    if name.startswith(folder_prefix) and not name.endswith('/'):
                        file_name = os.path.basename(name)
                        dst_file = dst_folder / f"{folder_name}_{file_name}"
                        with zf.open(name) as src, open(dst_file, "wb") as out:
                            shutil.copyfileobj(src, out)
                        copied += 1
                        total_extracted += 1
            print(f"[{split}] '{swahili_label}': {copied} images")

print(f"\nTotal files extracted: {total_extracted}")

os.remove(ZIP_PATH)
print("🗑️  Deleted fruits.zip — disk footprint is just the extracted subset above.")


## Step 4 — Load datasets (correctly, this time)

Pointing directly at the `Training/` and `Test/` subfolders — **not** the parent
`dataset_selected/` folder — so the fruit names are the classes, not `Training`/`Test`.

The assert right after is the sanity check that would have caught the original bug
before burning 3 hours on a bad run.


In [ ]:
TRAIN_DIR = OUTPUT_DIR / "Training"
TEST_DIR = OUTPUT_DIR / "Test"

train_set = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=(100, 100),
    batch_size=32,
    label_mode='categorical'
)

val_set = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=(100, 100),
    batch_size=32,
    label_mode='categorical'
)

test_set = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    seed=42,
    image_size=(100, 100),
    batch_size=32,
    label_mode='categorical',
    shuffle=False
)

class_names = train_set.class_names
num_classes = len(class_names)
print(f"Loaded {num_classes} classes: {class_names}")

# Sanity check: fail fast instead of training on garbage labels
assert "Training" not in class_names and "Test" not in class_names, \
    "class_names contains split-folder names, not fruit names — check TRAIN_DIR/TEST_DIR paths!"
assert num_classes > 2, f"Expected ~16 fruit classes, got {num_classes} — something's wrong."
print("✅ class_names look correct.")


## Step 5 — Preprocessing pipeline (MobileNetV2-compatible)

MobileNetV2 expects its own specific preprocessing (scaled to [-1, 1]), so we use
`preprocess_input` instead of the manual `Rescaling(1/255)` from the original notebook.


In [ ]:
preprocess = tf.keras.applications.mobilenet_v2.preprocess_input

train_set = train_set.map(lambda x, y: (preprocess(x), y))
val_set = val_set.map(lambda x, y: (preprocess(x), y))
test_set = test_set.map(lambda x, y: (preprocess(x), y))

AUTOTUNE = tf.data.AUTOTUNE

# Cache to DISK, not RAM — the merged dataset (all matched variety folders per
# fruit) is bigger than it looks, and Colab's free tier only has ~12-13GB RAM.
# Disk caching avoids the "session crashed after using all available RAM" issue.
os.makedirs("/content/tf_cache", exist_ok=True)
train_set = train_set.cache("/content/tf_cache/train").shuffle(buffer_size=200).prefetch(buffer_size=AUTOTUNE)
val_set = val_set.cache("/content/tf_cache/val").prefetch(buffer_size=AUTOTUNE)
test_set = test_set.cache("/content/tf_cache/test").prefetch(buffer_size=AUTOTUNE)

print("Dataset loading and preprocessing complete (disk-cached).")


## Step 6 — Model: MobileNetV2 transfer learning

Instead of training a 3-block CNN from scratch (slow to converge, needs lots of epochs),
we reuse a MobileNetV2 backbone pretrained on ImageNet and only train a small
classification head on top. This converges in a handful of epochs.


In [ ]:
from tensorflow.keras import layers, models

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(100, 100, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # freeze the pretrained backbone

model = models.Sequential([
    layers.Input(shape=(100, 100, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


## Step 7 — Train (few epochs — transfer learning converges fast)

In [ ]:
import time
from tensorflow.keras.callbacks import Callback
import matplotlib.pyplot as plt

class FormattedTrainingLogger(Callback):
    def __init__(self, model_name="Fruits Swahili MobileNetV2", total_epochs=15, print_every=1, patience=5):
        super().__init__()
        self.model_name = model_name
        self.total_epochs = total_epochs
        self.print_every = print_every
        self.patience = patience
        self.best_val_acc = 0.0
        self.best_weights = None
        self.patience_counter = 0
        self.start_time = None

    def on_train_begin(self, logs=None):
        self.start_time = time.time()
        device_name = "GPU" if tf.config.list_physical_devices('GPU') else "CPU"
        print(f"\nTraining {self.model_name} on {device_name}\n")

    def on_epoch_end(self, epoch, logs=None):
        current_epoch = epoch + 1
        train_loss = logs.get('loss', 0.0)
        val_loss = logs.get('val_loss', 0.0)
        val_acc = logs.get('val_accuracy', 0.0) * 100.0

        if val_acc > self.best_val_acc:
            self.best_val_acc = val_acc
            self.best_weights = self.model.get_weights()
            self.patience_counter = 0
        else:
            self.patience_counter += 1

        if current_epoch % self.print_every == 0 or current_epoch == self.total_epochs:
            print(f"Epoch [{current_epoch:02d}/{self.total_epochs}] | "
                  f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                  f"Val Acc: {val_acc:.2f}% | Best Val Acc: {self.best_val_acc:.2f}%")

        if self.patience_counter >= self.patience:
            print(f"Early stopping at epoch {current_epoch} (no improvement for {self.patience} epochs).")
            self.model.stop_training = True
            if self.best_weights is not None:
                self.model.set_weights(self.best_weights)

    def on_train_end(self, logs=None):
        elapsed = time.time() - self.start_time
        print(f"\n{self.model_name} training complete in {elapsed:.1f}s. Best Val Acc: {self.best_val_acc:.2f}%")

TOTAL_EPOCHS = 15
logger = FormattedTrainingLogger(total_epochs=TOTAL_EPOCHS, print_every=1, patience=5)

history = model.fit(
    train_set,
    validation_data=val_set,
    epochs=TOTAL_EPOCHS,
    callbacks=[logger],
    verbose=0
)

acc = history.history['accuracy']
val_acc_hist = history.history['val_accuracy']
loss = history.history['loss']
val_loss_hist = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc_hist, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss_hist, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Loss')

plt.tight_layout()
plt.show()


## Step 8 — Evaluate on the held-out Test split

In [ ]:
test_loss, test_acc = model.evaluate(test_set, verbose=0)
print(f"Test accuracy: {test_acc*100:.2f}%  |  Test loss: {test_loss:.4f}")


## Step 9 — Save model + correct class_indices.json

In [ ]:
import json

if logger.best_weights is not None:
    model.set_weights(logger.best_weights)
    print(f"✅ Restored best weights (Peak Val Acc: {logger.best_val_acc:.2f}%)")

MODEL_SAVE_PATH = "/content/fruits_cnn_swahili.keras"
CLASS_MAP_PATH = "/content/class_indices.json"

model.save(MODEL_SAVE_PATH)

index_to_swahili = {i: name for i, name in enumerate(class_names)}
with open(CLASS_MAP_PATH, "w") as f:
    json.dump(index_to_swahili, f, ensure_ascii=False, indent=2)

print(f"✅ Saved model to: {MODEL_SAVE_PATH}")
print(f"✅ Saved class map to: {CLASS_MAP_PATH}")
print(json.dumps(index_to_swahili, indent=2, ensure_ascii=False))


## Step 10 — Convert to TFLite (for your Streamlit app)

In [ ]:
model_for_export = tf.keras.models.load_model(MODEL_SAVE_PATH)

converter = tf.lite.TFLiteConverter.from_keras_model(model_for_export)
tflite_model = converter.convert()

TFLITE_PATH = "/content/fruits_cnn_swahili.tflite"
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

size_mb = os.path.getsize(TFLITE_PATH) / (1024 * 1024)
print(f"✅ Converted & saved: {TFLITE_PATH} ({size_mb:.1f} MB)")


## Step 11 — Download the two files your Streamlit app needs

This pops a browser download dialog for each file — no need to touch Google Drive.


In [ ]:
from google.colab import files

files.download("/content/fruits_cnn_swahili.tflite")
files.download("/content/class_indices.json")
